##  Ingest results.json files
### Read the file using spark dataframe reader API
### Add Meta Data Columns
- Source File
- Ingestion Timestamp
### Write to Bronze delta table

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-help

In [0]:
source_file=f"{landing_folder_path}/results"
table_name=f"{catalog_name}.{bronze_schema}.results"

###  Step 1- Read the csv file using dataframe reader API

In [0]:
#define the schema
from pyspark.sql.types import StructType,StructField,StringType,DateType,IntegerType,FloatType
results_schema=StructType([
    StructField('date',DateType()),
    StructField('raceName',StringType()),
    StructField('round',IntegerType()),
    StructField('season',IntegerType()),
     StructField('url',StringType()),
    StructField('constructorId',StringType()),
    StructField('driverId',StringType()),
   StructField('grid',IntegerType()),
    StructField('laps',IntegerType()),
    StructField('number',IntegerType()),
    StructField('points',FloatType()),
    StructField('position',IntegerType()),
    StructField('positionText',StringType()),
    StructField('status',StringType())
])

In [0]:
#spark reader API
results_df=(spark.read
                .format('json')
                .schema(results_schema)
                .option("mode","failFast")
                .load(source_file)
                )


### Step 2 Add Meta Data Columns
- Source File
- Ingestion Timestamp

In [0]:
results_final_df=add_ingestion_metadata(results_df)

### Step 3 Write to Bronze delta table

In [0]:
(
    results_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))

date,raceName,round,season,url,constructorId,driverId,grid,laps,number,points,position,positionText,status,ingestion_timestamp,source_file
1970-03-07,south african grand prix,1,1970,https://en.wikipedia.org/wiki/1970_South_African_Grand_Prix,brabham,jack_brabham,3,80,12,9.0,1,1,Finished,2026-10-06T17:48:53.135469Z,dbfs:/Volumes/formula1/landing/files/results/results_1970.json
1970-03-07,south african grand prix,1,1970,https://en.wikipedia.org/wiki/1970_South_African_Grand_Prix,mclaren-ford,hulme,6,80,6,6.0,2,2,Finished,2026-10-06T17:48:53.135469Z,dbfs:/Volumes/formula1/landing/files/results/results_1970.json
1970-03-07,south african grand prix,1,1970,https://en.wikipedia.org/wiki/1970_South_African_Grand_Prix,march,stewart,1,80,1,4.0,3,3,Finished,2026-10-06T17:48:53.135469Z,dbfs:/Volumes/formula1/landing/files/results/results_1970.json
1970-03-07,south african grand prix,1,1970,https://en.wikipedia.org/wiki/1970_South_African_Grand_Prix,matra,beltoise,8,80,3,3.0,4,4,Finished,2026-10-06T17:48:53.135469Z,dbfs:/Volumes/formula1/landing/files/results/results_1970.json
1970-03-07,south african grand prix,1,1970,https://en.wikipedia.org/wiki/1970_South_African_Grand_Prix,team_lotus,miles,14,79,10,2.0,5,5,+1 Lap,2026-10-06T17:48:53.135469Z,dbfs:/Volumes/formula1/landing/files/results/results_1970.json
1970-03-07,south african grand prix,1,1970,https://en.wikipedia.org/wiki/1970_South_African_Grand_Prix,team_lotus,hill,19,79,11,1.0,6,6,+1 Lap,2026-10-06T17:48:53.135469Z,dbfs:/Volumes/formula1/landing/files/results/results_1970.json
1970-03-07,south african grand prix,1,1970,https://en.wikipedia.org/wiki/1970_South_African_Grand_Prix,matra,pescarolo,18,78,4,0.0,7,7,+2 Laps,2026-10-06T17:48:53.135469Z,dbfs:/Volumes/formula1/landing/files/results/results_1970.json
1970-03-07,south african grand prix,1,1970,https://en.wikipedia.org/wiki/1970_South_African_Grand_Prix,team_lotus,love,22,78,23,0.0,8,8,+2 Laps,2026-10-06T17:48:53.135469Z,dbfs:/Volumes/formula1/landing/files/results/results_1970.json
1970-03-07,south african grand prix,1,1970,https://en.wikipedia.org/wiki/1970_South_African_Grand_Prix,brm,rodriguez,16,76,20,0.0,9,9,+4 Laps,2026-10-06T17:48:53.135469Z,dbfs:/Volumes/formula1/landing/files/results/results_1970.json
1970-03-07,south african grand prix,1,1970,https://en.wikipedia.org/wiki/1970_South_African_Grand_Prix,march,siffert,9,75,16,0.0,10,10,+5 Laps,2026-10-06T17:48:53.135469Z,dbfs:/Volumes/formula1/landing/files/results/results_1970.json


In [0]:
%sql
select season,count(*)
from formula1.bronze.results
group by season
order by season

season,count(1)
null,5
1950,160
1951,179
1952,204
1953,246
1954,229
1955,177
1956,189
1957,166
1958,226
